In [1]:
pip install feedparser requests beautifulsoup4

     |████████████████████████████████| 81 kB 3.0 MB/s eta 0:00:011
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
     |████████████████████████████████| 107 kB 7.1 MB/s eta 0:00:01
  Using cached idna-3.11-py3-none-any.whl (71 kB)
     |████████████████████████████████| 131 kB 14.6 MB/s eta 0:00:01
     |████████████████████████████████| 152 kB 5.0 MB/s eta 0:00:01
  Using cached charset_normalizer-3.4.4-cp39-cp39-macosx_10_9_universal2.whl (209 kB)
Using legacy 'setup.py install' for sgmllib3k, since package 'wheel' is not installed.
    Running setup.py install for sgmllib3k ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

RSS einlesen

In [ ]:
import feedparser

RSS_URL = "https://correctiv.org/feed/"

feed = feedparser.parse(RSS_URL)

print("entries:", len(feed.entries))


Einträge extrahieren

In [ ]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)


In [ ]:
print(rss_items)

Volltext aus den Artikeln holen

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

def fetch_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return None

    paragraphs = article.select("p")
    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text


In [ ]:
results = []

for item in rss_items:
    try:
        text = fetch_article(item["url"])
        item["text"] = text
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)


In [ ]:
import json

with open("../outputs/correctiv_articles.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


## Themenspezifisches Scraping

In [ ]:
import time
import requests
import feedparser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RSS_URL = "https://correctiv.org/faktencheck/tag/klima/feed/"

def make_session():
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    return s

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/121.0 Safari/537.36",
    "Accept": "application/rss+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}

session = make_session()

resp = session.get(RSS_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

feed = feedparser.parse(resp.text)

print("HTTP:", resp.status_code)
print("entries:", len(feed.entries))
print(feed.entries[0].get("title"), feed.entries[0].get("link"))


In [ ]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

def fetch_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return None

    paragraphs = article.select("p")
    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text


In [ ]:
results = []

for item in rss_items:
    try:
        text = fetch_article(item["url"])
        item["text"] = text
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)

In [ ]:
import json

with open("../outputs/correctiv_articles_klima.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Look at scraped articles

In [ ]:
path = "../outputs/correctiv_articles_klima.jsonl"
df = pd.read_json(path, lines=True)

df.head(10)

In [ ]:
import textwrap
from IPython.display import display, Markdown

df = pd.read_json("../outputs/correctiv_articles_klima.jsonl", lines=True)

# pick one article (by index or filter)
row = df.iloc[0]  # change index as needed

title = row.get("title", "Untitled")
text = row.get("text", "")

display(Markdown(f"## {title}\n\n{text}"))

In [ ]:
kw = "Fehlender Kontext"
hits = df[df["text"].str.contains(kw, case=False, na=False)]

hits[["title", "published", "url"]]

Achtung: das label (Faktencheck) scheint nicht erfasst zu sein,...